# Lesson 11b: Language Model Pretraining — Practical

11a derived the autoregressive objective, defined perplexity, and built
(and verified against PyTorch) a decoder-only mini-GPT's forward pass in
NumPy — forward pass only, no training. 10b already trained a
decoder-only Transformer in PyTorch, but only ever measured perplexity;
it never generated a single character of text. This notebook closes both
gaps at once: it trains 11a's exact architecture end to end, with a real
backward pass (PyTorch autograd, not derived by hand), on Tiny
Shakespeare, actually **samples text** from the trained model, puts that
sample next to one from a real pretrained GPT-2 checkpoint, and studies
how the sampling procedure itself — independent of training — reshapes
what comes out.

By the end of this notebook you will have:
- **trained** a decoder-only mini-GPT on Tiny Shakespeare for a bounded
  CPU budget, watching training loss and held-out perplexity fall,
- implemented **autoregressive sampling** (temperature and top-k) and
  generated text from the trained model,
- **compared generated samples** from the from-scratch tiny model against
  a real pretrained GPT-2 checkpoint, the same architecture family at a
  vastly different point on the parameters/data/compute axes 11a
  discussed, and
- shown, empirically, how **sampling parameters** (temperature, top-k)
  change generation quality and diversity from the *same* trained
  model and the *same* logits.

## Introduction

10b trained a decoder-only Transformer and reported a single number:
held-out perplexity. Perplexity measures how well the model predicts the
*next* token of *real* held-out text — it says nothing about what the
model produces when it has to generate text of its own, character by
character, with no ground truth to fall back on. **Generation** is a
genuinely different procedure from training: at training time every
position sees the true previous tokens (teacher forcing, 11a); at
generation time the model's own previous *output* becomes the next
input, and a sampling rule decides, at every step, which of the model's
predicted probabilities to actually commit to. This notebook trains
first (reusing 11a's architecture, this time all the way through a
backward pass), then generates, then asks a scale question 11a's
"Scaling Intuition" section could only gesture at: what does a
production pretrained model, sitting many orders of magnitude further
along the parameters/data/compute axes, actually sound like by
comparison?

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, minibatch sampling,
# generation) is reproducible.
import pathlib
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

### Tiny Shakespeare

The same corpus and character-level tokenisation 11a used, so the
architecture trained here sits on the identical data the theory notebook
already reasoned about.

In [ ]:
DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)
CORPUS_PATH = DATA_DIR / "tinyshakespeare.txt"
CORPUS_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not CORPUS_PATH.exists():
    urllib.request.urlretrieve(CORPUS_URL, CORPUS_PATH)
full_text = CORPUS_PATH.read_text(encoding="utf-8")

N_CHARS = 20000
text_slice = full_text[:N_CHARS]

chars = sorted(set(text_slice))
VOCAB_SIZE = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}


def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)


def decode(ids):
    return "".join(itos[int(i)] for i in ids)


split = int(0.9 * len(text_slice))
train_data = encode(text_slice[:split])
val_data = encode(text_slice[split:])
print(f"using the first {N_CHARS} characters, vocabulary: {VOCAB_SIZE} unique characters")
print(f"train: {len(train_data)} chars, val: {len(val_data)} chars")


def get_batch(data, seq_len, batch_size, rng):
    max_start = len(data) - seq_len - 1
    starts = rng.integers(0, max_start, size=batch_size)
    x = torch.stack([data[s:s + seq_len] for s in starts])
    y = torch.stack([data[s + 1:s + seq_len + 1] for s in starts])
    return x, y


def evaluate_perplexity(model, data, seq_len):
    model.eval()
    with torch.no_grad():
        x, y = data[:-1].unsqueeze(0), data[1:].unsqueeze(0)
        total_loss, total_len = 0.0, 0
        for start in range(0, x.shape[1], seq_len):
            xb, yb = x[:, start:start + seq_len], y[:, start:start + seq_len]
            if xb.shape[1] == 0:
                continue
            logits = model(xb)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1), reduction="sum")
            total_loss += loss.item()
            total_len += xb.shape[1]
    model.train()
    mean_nll = total_loss / total_len
    return mean_nll, float(np.exp(mean_nll))

## Training a Small GPT

The architecture is exactly 11a's decoder-only mini-GPT — token
embedding, positional embedding, stacked causal pre-norm Transformer
blocks, final layer norm, linear head — this time with a **learned**
positional embedding (10b's simpler alternative to 10a/11a's sinusoidal
formula, adequate here since every training sequence is well within the
fixed context length) and, critically, a real backward pass: PyTorch's
autograd differentiates straight through the cross-entropy loss 11a
derived, with no hand-written gradient anywhere in this notebook.

In [ ]:
def causal_mask(T):
    return torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))

    def forward(self, x, attn_mask):
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=attn_mask, need_weights=False)
        x = x + attn_out
        return x + self.ffn(self.ln2(x))


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_len):
        super().__init__()
        self.max_len = max_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        h = self.token_embed(x) + self.pos_embed(positions)[None, :, :]
        mask = causal_mask(T)
        for block in self.blocks:
            h = block(h, attn_mask=mask)
        h = self.ln_f(h)
        return self.head(h)


D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, SEQ_LEN = 64, 4, 3, 128, 64
BATCH_SIZE, N_ITERS, LR, GRAD_CLIP = 32, 500, 3e-3, 0.5

torch.manual_seed(SEED)
model = MiniGPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, max_len=SEQ_LEN)
n_params = sum(p.numel() for p in model.parameters())
print(f"MiniGPT: {NUM_LAYERS} layers, d_model={D_MODEL}, {NUM_HEADS} heads, {n_params:,} parameters")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
rng = np.random.default_rng(SEED)
train_losses, eval_iters, val_perplexities = [], [], []
for it in range(1, N_ITERS + 1):
    xb, yb = get_batch(train_data, SEQ_LEN, BATCH_SIZE, rng)
    logits = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
    optimizer.step()
    train_losses.append(loss.item())
    if it % 25 == 0 or it == 1:
        _, val_ppl = evaluate_perplexity(model, val_data, seq_len=SEQ_LEN)
        eval_iters.append(it)
        val_perplexities.append(val_ppl)

print(f"final train loss: {train_losses[-1]:.3f}")
print(f"final held-out perplexity: {val_perplexities[-1]:.2f}  (vocabulary size {VOCAB_SIZE})")

## Loss Curves and Samples

In [ ]:
fig, (ax_loss, ax_ppl) = plt.subplots(1, 2, figsize=(11, 4.5))
ax_loss.plot(train_losses)
ax_loss.set_xlabel("iteration")
ax_loss.set_ylabel("training cross-entropy loss")
ax_loss.set_title("Training loss")
ax_loss.grid(alpha=0.3)

ax_ppl.plot(eval_iters, val_perplexities, marker="o", color="tab:orange")
ax_ppl.axhline(VOCAB_SIZE, color="gray", linestyle="--", label="uniform-guess perplexity")
ax_ppl.set_xlabel("iteration")
ax_ppl.set_ylabel("held-out perplexity")
ax_ppl.set_title("Validation perplexity")
ax_ppl.legend()
ax_ppl.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Held-out perplexity falls well below the uniform-guessing baseline of
$V$ (11a's exact calibration point) — the model has genuinely learned
*something* about English/Shakespearean character statistics beyond
random guessing, from a training budget measured in seconds. Generation
now asks what that partial knowledge actually produces: starting from a
short prompt, repeatedly predict the next character's distribution,
sample one, append it, and feed the extended sequence back in — the
autoregressive factorisation from 11a, run forward at inference time
instead of used as a training target.

In [ ]:
def generate(model, prompt_ids, max_new_tokens, seq_len, temperature=1.0, top_k=None, seed=None):
    model.eval()
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    ids = prompt_ids.clone()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = ids[-seq_len:].unsqueeze(0)
            logits = model(context)[0, -1] / temperature
            if top_k is not None:
                top_values, _ = torch.topk(logits, min(top_k, logits.shape[-1]))
                logits = torch.where(logits < top_values[-1], torch.full_like(logits, -float("inf")), logits)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1, generator=g)
            ids = torch.cat([ids, next_id])
    model.train()
    return ids


PROMPT_TEXT = text_slice[:20]
prompt_ids = encode(PROMPT_TEXT)
print(f"prompt: {PROMPT_TEXT!r}")
print()
for sample_idx in range(3):
    generated = generate(model, prompt_ids, max_new_tokens=120, seq_len=SEQ_LEN,
                          temperature=0.8, top_k=20, seed=SEED + sample_idx)
    print(f"--- sample {sample_idx + 1} (temperature=0.8, top_k=20) ---")
    print(decode(generated))
    print()

The generated continuations are recognisably *character-level
Shakespearean-ish*: plausible letter and short-word statistics, roughly
the right mix of capitals, punctuation and line breaks, but not coherent
English past a word or two — exactly what a few hundred optimiser steps
over 18,000 training characters and a few hundred thousand parameters
should produce. "Comparison with Pretrained GPT-2" puts a number on how
much of the gap between this and fluent text is closed by orders of
magnitude more data, parameters and compute.

## Comparison with Pretrained GPT-2

`distilgpt2` is a real pretrained checkpoint — the same decoder-only
Transformer architecture family as the model just trained above, and
the same next-token cross-entropy objective 11a derived, but trained on
roughly 40GB of web text with roughly 82 million parameters, versus this
notebook's few-hundred-thousand-parameter model trained on 18,000
characters. Generating from both, from the *same* text prompt, makes the
scale gap directly comparable rather than an abstract number.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GPT2_NAME = "distilgpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_NAME)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_NAME)
gpt2_model.eval()

gpt2_n_params = sum(p.numel() for p in gpt2_model.parameters())
print(f"{GPT2_NAME}: {gpt2_n_params:,} parameters (this notebook's MiniGPT: {n_params:,})")

gpt2_prompt = "ROMEO:"
gpt2_inputs = gpt2_tokenizer(gpt2_prompt, return_tensors="pt")
torch.manual_seed(SEED)
with torch.no_grad():
    gpt2_output = gpt2_model.generate(
        **gpt2_inputs, max_new_tokens=40, do_sample=True, temperature=0.8, top_k=20,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )
gpt2_text = gpt2_tokenizer.decode(gpt2_output[0], skip_special_tokens=True)

mini_gpt_sample = decode(generate(model, encode(gpt2_prompt), max_new_tokens=40, seq_len=SEQ_LEN,
                                   temperature=0.8, top_k=20, seed=SEED))

print(f"prompt: {gpt2_prompt!r}\n")
print(f"MiniGPT ({n_params:,} params, trained here):\n  {mini_gpt_sample!r}\n")
print(f"{GPT2_NAME} ({gpt2_n_params:,} params, pretrained on web-scale text):\n  {gpt2_text!r}")

`distilgpt2`'s continuation is fluent, grammatical English — full words,
plausible syntax, a coherent register — while the from-scratch MiniGPT's
output is, at best, locally plausible character statistics. Both models
are decoder-only Transformers trained on the identical objective 11a
derived; nothing about the *architecture* or the *loss function*
explains the gap. What differs is exactly the three axes 11a's "Scaling
Intuition" section discussed: parameters (roughly 82M vs. a few hundred
thousand), data (tens of gigabytes of web text vs. 18,000 characters of
one play), and the compute spent combining the two — this comparison is
the concrete, qualitative payoff of the scaling argument 11a could only
make quantitatively at toy scale.

## Sampling Parameters

Everything above used one fixed decoding rule (temperature 0.8,
top-k 20). Decoding is a choice made *after* training, from the *same*
trained model and the *same* logits — it changes nothing about what the
model learned, only how its predicted distribution is turned into a
committed sequence of characters. **Temperature** rescales the logits
before the softmax ($\text{logits}/T$): $T<1$ sharpens the distribution
toward its mode (more deterministic, more repetitive), $T>1$ flattens it
(more diverse, more likely to wander into low-probability, less coherent
territory). **Top-k** truncates the distribution to only the $k$ most
likely next characters before sampling, regardless of temperature —
a hard floor against ever committing to a very-low-probability
character, however high the temperature.

In [ ]:
print(f"prompt: {PROMPT_TEXT!r}\n")
print("=== varying temperature (top_k=20 fixed) ===")
for temperature in [0.3, 0.8, 1.5]:
    sample = decode(generate(model, prompt_ids, max_new_tokens=80, seq_len=SEQ_LEN,
                              temperature=temperature, top_k=20, seed=SEED))
    print(f"T={temperature:>4}: {sample!r}")

print()
print("=== varying top_k (temperature=1.0 fixed) ===")
for top_k in [1, 5, VOCAB_SIZE]:
    sample = decode(generate(model, prompt_ids, max_new_tokens=80, seq_len=SEQ_LEN,
                              temperature=1.0, top_k=top_k, seed=SEED))
    label = "greedy" if top_k == 1 else ("no restriction" if top_k == VOCAB_SIZE else str(top_k))
    print(f"top_k={label:>15}: {sample!r}")

Low temperature collapses toward the same few high-probability
characters, often repeating short fragments almost verbatim; high
temperature produces visibly more varied, and less locally plausible,
character sequences. `top_k=1` is exactly greedy decoding — always the
single most likely next character, fully deterministic given the prompt
— while `top_k=`vocabulary size imposes no restriction at all, leaving
temperature as the only lever. Every one of these samples comes from
exactly the same trained weights and the same forward pass; the
differences are entirely a property of the decoding rule.

## Key Takeaways

- **11a's mini-GPT architecture trains end to end with ordinary
  PyTorch autograd**: no hand-derived backward pass was needed anywhere
  in this notebook, and held-out perplexity fell well below the
  uniform-guessing baseline within a few hundred iterations.
- **Generation is a different procedure from training**: it feeds the
  model's own sampled output back in as the next input, using the
  autoregressive factorisation forward rather than as a training target,
  and the from-scratch model's samples are locally plausible but not
  coherent English at this training budget.
- **A real pretrained GPT-2 checkpoint (`distilgpt2`) produces fluent,
  grammatical continuations from the same prompt**, despite sharing the
  identical architecture and objective — the gap is a direct, qualitative
  measurement of what 11a's parameters/data/compute scaling axes buy in
  practice.
- **Temperature and top-k reshape generation from the same trained
  model and the same logits**: low temperature and small top-k collapse
  toward repetitive, near-deterministic output; high temperature and
  large top-k increase diversity at the cost of local coherence — a
  decoding-time choice, entirely separate from training.